# Yagona variant: isbotlanadigan chegaralar, raundlar soni, bayt darajasidagi 128 bitli rejim va tezlik

Yagona variant — ikkala rejim bayt darajasida (4×8 va 4×4 bayt), Bog‘lash, SBZ, toq kₑ, holatdan mustaqil kalit jadvali
(RCⱼ — √2 kasr qismi), oldinga uzatish va P(H) ⊕ H (C₁, C₂ — √3 kasr qismi), raundlar soni R (yutish nuqtalari 0, R−2, R−1).
Daftar mustaqil ishlaydi (faqat `numpy`); C o‘lchovi uchun gcc/clang (Windows da MinGW) kerak.

In [1]:
# 0-hujayra. Dastur modullarini yuklash (daftar mustaqil ishlaydi).
# Agar modullar shu papkada bo'lsa, ular ishlatiladi; bo'lmasa quyidagi matndan xotirada yaratiladi.
import sys, types, importlib.util
_MODULLAR = {
    'algoritm': '"""\nO‘zMSt 285:2024, 1-algoritm: vektorlashtirilgan realizatsiya.\nVariantlar: p = 256 yoki p = 16; yutish o‘zgartirishi bz = \'std\' (BaytZichlash) yoki \'sbz\' (S(h ⊕ S(n))).\nBarcha funksiyalar (N, 32) o‘lchamli numpy massivlar ustida ishlaydi: bir vaqtda N ta mustaqil hisob.\n"""\nimport numpy as np\n\n# B ilova (majburiy): 1x256 chiziqli zichlash massivi (satr — yuqori yarim bayt)\nZICHLASH = bytes.fromhex(\n    "050d060b010a0f0800040709020c030e"\n    "0807020e0f030b06010c0d0a05040900"\n    "0e020d040c07010b06090005030a080f"\n    "000e090c030d07040f0605010b020a08"\n    "030a0702040c09010e0d0f0800050b06"\n    "02030108000e05090c0b06070a0f0d04"\n    "0a040e0f090508020b0001030c06070d"\n    "0b090a0106040d0f03050e000807020c"\n    "010003070d0b0a0c090e04060f080502"\n    "04080b090e0602050a030c0f070d0001"\n    "090c0f0002010e0a05080b0d04030607"\n    "060b080d07090003040f0a020e010c05"\n    "0f0100050a0803070d02090c060e040b"\n    "0c05040a0b02060d0807030e01000f09"\n    "070f0c060500040e020a080b0d090103"\n    "0d060503080f0c0007010204090b0e0a")\nZ = np.array(list(ZICHLASH), dtype=np.int64)\ndef aes_sbox():\n    s=[0]*256; p=q=1\n    while True:\n        p ^= (p<<1) ^ (0x1b if p&0x80 else 0); p&=0xff\n        q ^= q<<1; q ^= q<<2; q ^= q<<4; q&=0xff\n        if q&0x80: q^=0x09\n        x=q ^ ((q<<1|q>>7)&0xff) ^ ((q<<2|q>>6)&0xff) ^ ((q<<3|q>>5)&0xff) ^ ((q<<4|q>>4)&0xff)\n        s[p]=x^0x63\n        if p==1: break\n    s[0]=0x63; return np.array(s,dtype=np.int64)\nS=aes_sbox()\nclass Cfg:\n    def __init__(s,p=256,bz=\'sbz\'):\n        s.p=p; s.bz=bz\n        s.INV=np.zeros(p,dtype=np.int64)\n        for x in range(1,p,2): s.INV[x]=pow(x,-1,p)\ndef prododd(a,p):\n    b=np.where(a%2==0,a+1,a)%p; r=np.ones(a.shape[0],dtype=np.int64)\n    for i in range(a.shape[1]): r=(r*b[:,i])%p\n    return r\ndef dK(P,Jo,p): return np.where((P==(Jo+1)%p)|(P==(-(Jo+1))%p),10,0)\ndef step(x,A,B,R,c):\n    d=(B+R*x)%c.p; return (-A*x*c.INV[d])%c.p\ndef qosh(h,hn,R,c):\n    p=c.p; R=R[:,None]\n    J1h=h.sum(1)%p; J1hn=hn.sum(1)%p\n    Ph=prododd(h,p); Phn=prododd(hn,p)\n    K1h=(Ph+dK(Ph,J1hn,p))%p; K1hn=(Phn+dK(Phn,J1h,p))%p\n    dAh=np.where(J1hn%2==1,0,1); dAhn=np.where(J1h%2==1,0,1)\n    A1h=(J1hn[:,None]+2*hn+dAh[:,None])%p; A1hn=(J1h[:,None]+2*h+dAhn[:,None])%p\n    h1=step(h,A1h,(K1h[:,None]-2*h)%p,R,c); hn1=step(hn,A1hn,(K1hn[:,None]-2*hn)%p,R,c)\n    J2h=h1.sum(1)%p; J2hn=hn1.sum(1)%p\n    P2h=prododd(h1,p); P2hn=prododd(hn1,p)\n    K2h=(P2h+dK(P2h,J2hn,p))%p; K2hn=(P2hn+dK(P2hn,J2h,p))%p\n    dA2h=np.where((J1hn+J2hn)%2==1,0,1); dA2hn=np.where((J1h+J2h)%2==1,0,1)\n    A2h=(J1hn[:,None]+J2hn[:,None]+2*hn1+dA2h[:,None])%p; A2h=np.where(A2h==0,1,A2h)\n    A2hn=(J1h[:,None]+J2h[:,None]+2*h1+dA2hn[:,None])%p; A2hn=np.where(A2hn==0,1,A2hn)\n    B2h=((K1h*K2h)[:,None]-2*h1)%p; B2hn=((K1hn*K2hn)[:,None]-2*hn1)%p\n    h2=step(h1,A2h,B2h,R,c); h2=np.where(h2==0,(A2h+B2h)%p,h2)\n    hn2=step(hn1,A2hn,B2hn,R,c); hn2=np.where(hn2==0,(A2hn+B2hn)%p,hn2)\n    return step(h2,A2h,B2h,R,c), step(hn2,A2hn,B2hn,R,c)\ndef bzf(h,hn,c):\n    if c.p==256:\n        if c.bz==\'std\': return (Z[h]<<4)|Z[hn]\n        return S[h^S[hn]]\n    if c.bz==\'std\': return h^hn\n    H=16*h[:,0::2]+h[:,1::2]; N=16*hn[:,0::2]+hn[:,1::2]; Q=S[H^S[N]]\n    out=np.empty_like(h); out[:,0::2]=Q>>4; out[:,1::2]=Q&15; return out\ndef dia(H,K,p):  # H,K: (N,4,4)\n    sk=K.sum(1)            # sum_i K[i,u]  -> (N,4)\n    sh=H.sum(1)            # sum_i H[i,u]\n    R=np.empty_like(H)\n    for s in range(4):\n        for u in range(4):\n            if s==u:\n                v=H[:,u,u]*sk[:,u]-sum(H[:,i,i]*K[:,i,u] for i in range(4) if i!=u)\n            else:\n                v=H[:,s,u]*sk[:,u]+K[:,s,u]*sh[:,u]-sum(H[:,s,i]*K[:,i,u] for i in range(4) if i not in (s,u))\n            R[:,s,u]=v%p\n    return R\ndef aralash(h,ke,c):\n    H=h.reshape(-1,4,8); K=ke.reshape(-1,4,8)\n    L=dia(H[:,:,:4],K[:,:,:4],c.p); Rr=dia(H[:,:,4:],K[:,:,4:],c.p)\n    return np.concatenate([L,Rr],axis=2).reshape(-1,32)\ndef _shift(h,rows,cols):\n    m=h.reshape(-1,4,8).copy()\n    for s in range(4): m[:,s,:]=np.roll(m[:,s,:],rows[s],axis=1)\n    for u,n in cols.items(): m[:,:,u]=np.roll(m[:,:,u],n,axis=1)\n    return m.reshape(-1,32)\ndef surholat(h): return _shift(h,[1,2,3,4],{0:1,4:1,1:2,5:2,2:3,6:3})\ndef surkalit(ke): return _shift(ke,[4,3,2,1],{1:-3,5:-3,2:-2,6:-2,3:-1,7:-1})\ndef tuz(ke,k,c):\n    p=c.p\n    return np.where((ke==0)&(k%2==0),2,np.where((ke!=0)&(ke%2==k%2),ke,(ke-1)%p))\ndef iteration(holat,blok,ke,k,c,trace=None,rounds=10):\n    """Bitta tashqi iteratsiya (i): Qo\'sh+BZ, so\'ng \'rounds\' ta bosqich."""\n    R=k[:,0]\n    def T(name,j,h,kk):\n        if trace is not None: trace.append((name,j,h.copy(),kk.copy()))\n    holat,hn=qosh(holat,blok,R,c); T(\'Qosh0\',0,holat,ke)\n    holat=bzf(holat,hn,c); T(\'BZ0\',0,holat,ke)\n    for j in range(1,rounds+1):\n        holat=aralash(holat,ke,c); T(\'Aralash\',j,holat,ke)\n        holat,ke=qosh(holat,ke,R,c); T(\'Qosh1\',j,holat,ke)\n        holat=surholat(holat); ke=surkalit(ke); T(\'Sur\',j,holat,ke)\n        if j==8:\n            holat,hn=qosh(holat,blok,R,c); T(\'Qosh_inj8\',j,holat,ke); holat=bzf(holat,hn,c); T(\'BZ8\',j,holat,ke)\n        else:\n            holat,ke=qosh(holat,ke,R,c); T(\'Qosh2\',j,holat,ke)\n        ke=tuz(ke,k,c); T(\'Tuz\',j,holat,ke)\n        if j==9:\n            holat,hn=qosh(holat,blok,R,c); T(\'Qosh_inj9\',j,holat,ke); holat=bzf(holat,hn,c); T(\'BZ9\',j,holat,ke)\n    return holat,ke\ndef to_units(b,p):\n    b=np.asarray(b,dtype=np.int64)\n    if p==256: return b\n    out=np.empty(b.shape[:-1]+(b.shape[-1]*2,),dtype=np.int64); out[...,0::2]=b>>4; out[...,1::2]=b&15; return out\ndef hash_msgs(msgs,keys,c):\n    """msgs: (N,L) bayt, L - blok uzunligiga karrali; keys: (N,32 yoki 16)."""\n    return _hash_core(msgs,keys,c,bits=msgs.shape[1]*8)\ndef _hash_core(msgs,keys,c,bits):\n    N,L=msgs.shape; bl=32 if c.p==256 else 16\n    nb=L//bl\n    blocks=[msgs[:,i*bl:(i+1)*bl] for i in range(nb)]\n    ny=[0]*N\n    nyb=np.zeros((N,bl),dtype=np.int64)\n    for n in range(N):\n        v=sum(int.from_bytes(bytes(b[n].tolist()),\'big\') for b in blocks)%(2**(8*bl))\n        nyb[n]=list(v.to_bytes(bl,\'big\'))\n    lb=np.tile(np.array(list(bits.to_bytes(bl,\'big\'))),(N,1))\n    kir=[to_units(b,c.p) for b in blocks]+[to_units(lb,c.p),to_units(nyb,c.p)]\n    k=to_units(keys,c.p); ke=k.copy(); holat=kir[0]\n    for i in range(1,len(kir)): holat,ke=iteration(holat,kir[i],ke,k,c)\n    return holat\n\n\n# ------------------------------------------------------------------ nazorat qiymatlari\nKAT_KALIT = bytes.fromhex("c86ed4b2756df1cd6b6eca81d441809221ab81bba0c7c15044be8feb934f95a2")\nKAT_XABAR = b"funksiyaxeshirovaniyainformasionnayatexnologiyakriptograficheska"\nKAT = {\n    (256, \'std\'): "55394f2d84fccd4d8d01d8fef756cd657ebecc3f96f81e5207767447de66e2b6",\n    (256, \'sbz\'): "42f97daaa8e7010bc629ca695eaa77df7ae7443a0c3ae78d205c2e025d256c96",\n    (16, \'std\'):  "e9eaccdd4ccce2d229228c864989eb9f",\n    (16, \'sbz\'):  "bd745e2ed572ea4246ed68482ef46c5e",\n}\n\ndef xesh(xabar: bytes, kalit: bytes, p=256, bz=\'sbz\') -> bytes:\n    """Bitta xabar uchun xesh qiymat (xabar uzunligi blok uzunligiga karrali bo‘lishi shart emas)."""\n    bl = 32 if p == 256 else 16\n    m = xabar + b"\\x00" * (-len(xabar) % bl) if xabar else b"\\x00" * bl\n    # uzunlik bloki asl xabar uzunligi bo‘yicha hisoblanadi\n    c = Cfg(p, bz)\n    msgs = np.array([list(m)]); keys = np.array([list(kalit[:bl])])\n    out = _hash_core(msgs, keys, c, bits=len(xabar) * 8)[0]\n    if p == 16: out = [out[2*i] << 4 | out[2*i+1] for i in range(16)]\n    return bytes(int(x) for x in out)\n\ndef nazorat():\n    """A ilova va SBZ materialidagi nazorat qiymatlarini tekshiradi."""\n    ok = True\n    for (p, bz), kut in KAT.items():\n        r = xesh(KAT_XABAR, KAT_KALIT, p, bz).hex()\n        print(f"  p={p:3d}, {bz}: {\'MOS\' if r == kut else \'MOS EMAS\'}")\n        ok &= (r == kut)\n    return ok\n\nif __name__ == "__main__":\n    print("Nazorat qiymatlari:"); print("Hammasi mos" if nazorat() else "XATO")\n',
    'komponentlar': '"""Kriptotahlil uchun umumiy yordamchi funksiyalar."""\nimport itertools\nimport numpy as np\nfrom algoritm import Cfg, aralash, surholat\n\nPAR8 = np.array([bin(i).count(\'1\') & 1 for i in range(256)])\nX256 = np.arange(256)\n\ndef kalitlar(rng, N, p):\n    """Tasodifiy kalitlar: R (1-element) va 2-element juft (5.1.2-band talabi)."""\n    if p == 256:\n        k = rng.integers(0, 256, (N, 32)); k[:, 0] &= 0xFE; k[:, 1] &= 0xFE\n    else:\n        k = rng.integers(0, 16, (N, 32)); k[:, 0] &= 0xE; k[:, 1] &= 0xE\n    return k\n\ndef wht(f):\n    """Walsh–Adamar o‘zgartirishi (f — ±1 vektor)."""\n    a = f.astype(np.int64).copy(); h = 1; n = len(a)\n    while h < n:\n        a = a.reshape(-1, 2, h); a = np.stack([a[:, 0] + a[:, 1], a[:, 0] - a[:, 1]], 1).reshape(-1); h *= 2\n    return a\n\ndef lat(F, nout):\n    """8 bitli kirishli F uchun W[β][α] = Σ (−1)^(β·F(x) ⊕ α·x); korrelyatsiya c = W/256."""\n    W = np.zeros((1 << nout, 256), dtype=np.int64)\n    for b in range(1 << nout):\n        W[b] = wht(1 - 2 * PAR8[F & b])\n    return W\n\ndef ddt(F, nout):\n    """Ayirmalar matritsasi DDT[Δx][Δy] (8 bitli kirish)."""\n    return np.array([np.bincount(F ^ F[X256 ^ d], minlength=nout) for d in range(256)])\n\ndef anf_deg(fbits, nv):\n    """Boolean funksiyaning ANF darajasi (Möbius o‘zgartirishi)."""\n    a = fbits.astype(np.uint8).copy(); h = 1; N = 1 << nv\n    while h < N:\n        a = a.reshape(-1, 2, h); a[:, 1] ^= a[:, 0]; a = a.reshape(-1); h *= 2\n    wt = np.array([bin(i).count(\'1\') for i in range(N)]); nz = wt[a == 1]\n    return int(nz.max()) if len(nz) else -1\n\ndef bits(v, n): return (v[:, None] >> np.arange(n)) & 1\n\ndef implicit_count(points, deg):\n    """deg darajali chiziqli bog‘liq bo‘lmagan implitsit tenglamalar soni (monomlar matritsasi rangi orqali)."""\n    npts, nv = points.shape\n    mons = [()] + [m for d in range(1, deg + 1) for m in itertools.combinations(range(nv), d)]\n    words = (npts + 63) // 64; cols = []\n    for m in mons:\n        v = np.ones(npts, dtype=np.uint8)\n        for i in m: v &= points[:, i].astype(np.uint8)\n        pad = np.zeros(words * 64, dtype=np.uint8); pad[:npts] = v\n        cols.append(np.packbits(pad).view(np.uint64))\n    A = np.array(cols); rank = 0; n = A.shape[0]\n    for w in range(A.shape[1]):\n        for b in range(64):\n            if rank >= n: break\n            mask = np.uint64(1) << np.uint64(b)\n            idx = np.nonzero((A[rank:, w] & mask) != 0)[0]\n            if len(idx) == 0: continue\n            piv = rank + idx[0]\n            if piv != rank: A[[rank, piv]] = A[[piv, rank]]\n            rows = np.nonzero((A[:, w] & mask) != 0)[0]; rows = rows[rows != rank]\n            A[rows] ^= A[rank]; rank += 1\n    return len(mons) - rank, len(mons)\n\ndef qosh_elem(A1, B1, A2, B2, R, p=256):\n    """Qo‘sh o‘zgartirishining bitta elementi (parametrlar belgilangan): x → x^(−3)."""\n    x = np.arange(p); INV = np.zeros(p, dtype=np.int64)\n    for t in range(1, p, 2): INV[t] = pow(t, -1, p)\n    s = lambda x, A, B: (-A * x * INV[(B + R * x) % p]) % p\n    y1 = s(x, A1, B1); y2 = s(y1, A2, B2); y2 = np.where(y2 == 0, (A2 + B2) % p, y2)\n    return s(y2, A2, B2)\n\ndef tasodifiy_param(rng):\n    A1, B1, A2, B2 = [int(rng.integers(0, 128)) * 2 + 1 for _ in range(4)]\n    return A1, B1, A2, B2, int(rng.integers(0, 128)) * 2\n\ndef rank2(A):\n    A = A.copy() % 2; r = 0\n    for col in range(A.shape[1]):\n        piv = [i for i in range(r, A.shape[0]) if A[i, col]]\n        if not piv: continue\n        A[[r, piv[0]]] = A[[piv[0], r]]\n        for i in range(A.shape[0]):\n            if i != r and A[i, col]: A[i] ^= A[r]\n        r += 1\n    return r\n\n# 0-qatlam matritsalari: SurHolat (P) va Aralash (M(k0))\n_C2 = Cfg(2, \'std\')\n_perm = surholat(np.arange(32)[None, :].copy())[0]\nP_SUR = np.zeros((32, 32), dtype=np.int64); P_SUR[np.arange(32), _perm] = 1\n\ndef M_aralash(k):\n    """h0\' = M(k0)·h0 (mod 2) matritsasi."""\n    E = np.eye(32, dtype=np.int64); K = np.tile(np.asarray(k) % 2, (32, 1))\n    return (aralash(E, K, _C2).T) % 2\n\ndef L_raund(k):\n    """Yutishsiz raundning 0-qatlam matritsasi L = P·M(k0)."""\n    return (P_SUR @ M_aralash(k)) % 2\n',
    'algebraik': '"""\n3-bo‘lim. Algebraik kriptotahlil.\n  1) implitsit tenglamalar soni va ANF darajalari: S, Z, SBZ, BaytZichlash-256, Qo‘sh elementi;\n  2) kub testlari: r = 0…3 raunddan keyin daraja o‘sishi;\n  3) hujum namoyishi: chiqish 0-qatlamini xabar 0-qatlamining affin funksiyasi sifatida tiklash;\n  4) bit darajasidagi tenglamalar soni (baho modeli) va linearizatsiya murakkabligi.\nNatija: natijalar/algebraik.json\n"""\nimport argparse, json, math, os\nimport numpy as np\nfrom algoritm import S, Z, Cfg, iteration, hash_msgs, to_units\nfrom komponentlar import implicit_count, anf_deg, bits, qosh_elem, tasodifiy_param, kalitlar\n\ndef tenglamalar_va_daraja(rng, trials=50):\n    R = {}; x = np.arange(256)\n    R[\'S_kvadrat\'] = implicit_count(np.concatenate([bits(x, 8), bits(S[x], 8)], 1), 2)\n    R[\'Z_kvadrat\'] = implicit_count(np.concatenate([bits(x, 8), bits(Z[x], 4)], 1), 2)\n    R[\'Z_kub\'] = implicit_count(np.concatenate([bits(x, 8), bits(Z[x], 4)], 1), 3)\n    f = qosh_elem(*tasodifiy_param(rng))\n    R[\'Qosh_kvadrat\'] = implicit_count(np.concatenate([bits(x, 8), bits(f, 8)], 1), 2)\n    R[\'Qosh_kub\'] = implicit_count(np.concatenate([bits(x, 8), bits(f, 8)], 1), 3)\n    hh = np.arange(65536) >> 8; nn = np.arange(65536) & 255\n    y = S[hh ^ S[nn]]; yb = (Z[hh] << 4) | Z[nn]\n    R[\'SBZ_kvadrat\'] = implicit_count(np.concatenate([bits(hh, 8), bits(nn, 8), bits(y, 8)], 1), 2)\n    R[\'BZ256_kvadrat\'] = implicit_count(np.concatenate([bits(hh, 8), bits(nn, 8), bits(yb, 8)], 1), 2)\n    R[\'S_daraja\'] = [anf_deg((S >> j) & 1, 8) for j in range(8)]\n    R[\'Z_daraja\'] = [anf_deg((Z >> j) & 1, 8) for j in range(4)]\n    R[\'SBZ_daraja\'] = [anf_deg((y >> j) & 1, 16) for j in range(8)]\n    R[\'BZ256_daraja\'] = [anf_deg((yb >> j) & 1, 16) for j in range(8)]\n    D = np.array([[anf_deg((qosh_elem(*tasodifiy_param(rng)) >> j) & 1, 8) for j in range(8)] for _ in range(trials)])\n    R[\'Qosh_daraja_min\'] = D.min(0).tolist(); R[\'Qosh_daraja_mediana\'] = np.median(D, 0).tolist(); R[\'Qosh_daraja_max\'] = D.max(0).tolist()\n    return R\n\ndef kub_testlari(rng, dlar=(2, 4, 6, 8, 10, 12), raundlar=(0, 1, 2, 3), ncubes=4):\n    """Xabarning dastlabki 16 bitidan d ta bit; 4 ta kub bo‘yicha yig‘indisi doim nol bo‘lgan bitlar ulushi."""\n    R = {}\n    for p, bz in [(256, \'std\'), (256, \'sbz\'), (16, \'std\'), (16, \'sbz\')]:\n        c = Cfg(p, bz); ub = 8 if p == 256 else 4\n        for r in raundlar:\n            row = {}\n            for d in dlar:\n                k = kalitlar(rng, 1, p)[0]; h = rng.integers(0, p, 32); bl = rng.integers(0, p, 32); allz = None\n                for _ in range(ncubes):\n                    pos = rng.choice(16, d, replace=False); base = bl.copy(); base[:16 // ub] = rng.integers(0, p, 16 // ub)\n                    N = 1 << d; Bm = np.tile(base, (N, 1))\n                    for t, q in enumerate(pos):\n                        u, b = q // ub, q % ub; Bm[:, u] &= ~(1 << b); Bm[:, u] |= ((np.arange(N) >> t) & 1) << b\n                    K = np.tile(k, (N, 1)); out, _ = iteration(np.tile(h, (N, 1)), Bm, K.copy(), K, c, rounds=r)\n                    s = np.bitwise_xor.reduce(out, axis=0); zb = np.array([(s >> j) & 1 for j in range(ub)]).T == 0\n                    allz = zb if allz is None else (allz & zb)\n                row[str(d)] = {\'barcha_bitlar\': round(float(allz.mean()), 3), \'0-qatlam\': round(float(allz[:, 0].mean()), 3)}\n            R[f\'{p}_{bz}_r{r}\'] = row\n            print(f"  p={p}, {bz}, r={r}: " + ", ".join(f"d={d}:{v[\'barcha_bitlar\']}" for d, v in row.items()))\n    return R\n\ndef gf2_solve(X, Y):\n    A = np.concatenate([X, Y], 1).astype(np.uint8) % 2; m, n = X.shape; r = 0; piv = []\n    for col in range(n):\n        idx = [i for i in range(r, m) if A[i, col]]\n        if not idx: continue\n        A[[r, idx[0]]] = A[[idx[0], r]]\n        for i in range(m):\n            if i != r and A[i, col]: A[i] ^= A[r]\n        piv.append(col); r += 1\n    W = np.zeros((n, Y.shape[1]), dtype=np.uint8)\n    for i, col in enumerate(piv): W[col] = A[i, n:]\n    return W\n\ndef linearizatsiya_hujumi(rng, n_oquv=80, n_sinov=300):\n    R = {}\n    for p, bz in [(16, \'std\'), (16, \'sbz\'), (256, \'std\'), (256, \'sbz\')]:\n        c = Cfg(p, bz); bl = 32 if p == 256 else 16\n        if p == 256: k = rng.integers(0, 256, 32); k[0] &= 0xFE; k[1] &= 0xFE\n        else: k = rng.integers(0, 256, 16); k[0] &= 0xEE\n        H = lambda m: hash_msgs(m, np.tile(k, (len(m), 1)), c)\n        feats = lambda m: np.concatenate([to_units(m, p) % 2, np.ones((len(m), 1), dtype=np.int64)], 1)\n        tr = rng.integers(0, 256, (n_oquv, bl)); te = rng.integers(0, 256, (n_sinov, bl))\n        W = gf2_solve(feats(tr), H(tr) % 2); pred = (feats(te) @ W) % 2; act = H(te) % 2\n        R[f\'{p}_{bz}\'] = {\'toliq_32_bit\': float((pred == act).all(1).mean()), \'bitlar_boyicha\': round(float((pred == act).mean()), 4)}\n        print(f"  p={p}, {bz}: {R[f\'{p}_{bz}\']}")\n    return R\n\ndef tenglamalar_soni():\n    """Bit darajasidagi baho modeli: (o‘zgaruvchilar, tenglamalar, nochiziqli tenglamalar)."""\n    def counts(n, mode):\n        w = 32; ADD = (2*n-1, 2*n-1, n-1); MUL = (n*(n+1)//2 + (n-1)*(2*n-1),) * 2 + (n*(n+1)//2 + (n-1)*(n-1),)\n        add = lambda a, b, f=1: tuple(x + f*y for x, y in zip(a, b)); z = (0, 0, 0); q = z\n        for _ in range(2):\n            q = add(q, ADD, (w-1)*2); q = add(q, MUL, (w-1)*2); q = add(q, (2*n,)*3, 4); q = add(q, MUL, 1)\n            q = add(q, ADD, w*4); q = add(q, MUL, w*9); q = add(q, ADD, w*6); q = add(q, (n,)*3, w)\n        ar = add(add(z, MUL, 128), ADD, 144); tz = add(add(z, ADD, w), (n,)*3, w)\n        if mode == \'sbz\': nb = w if n == 8 else w // 2; bz = (16*nb, 78*nb, 78*nb)\n        elif n == 8: bz = (8*w, 43*2*w + 8*w, 43*2*w)\n        else: bz = (0, n*w, 0)\n        rnd = add(add(ar, q, 2), tz); r0 = add(q, bz)\n        out = dict(qosh=q, aralash=ar, tuzilma=tz, yutish=bz, raund=rnd, r1=add(r0, rnd))\n        out[\'r2\'] = add(out[\'r1\'], rnd); out[\'r3\'] = add(out[\'r2\'], rnd); return out\n    E = {f\'{lab}_{m}\': counts(n, m) for n, lab in [(8, \'256\'), (4, \'128\')] for m in (\'std\', \'sbz\')}\n    lc = lambda nv, d: round(2.8 * math.log2(sum(math.comb(nv, i) for i in range(d + 1))), 1)\n    return E, {f\'{nv}_ozg_d{d}\': lc(nv, d) for nv in (128, 256) for d in (1, 4, 6, 7, 8, 13)}\n\nif __name__ == "__main__":\n    ap = argparse.ArgumentParser(); ap.add_argument(\'--tez\', action=\'store_true\')\n    a = ap.parse_args(); os.makedirs(\'natijalar\', exist_ok=True); out = {}\n    print("1. Implitsit tenglamalar va darajalar..."); out[\'tenglamalar_daraja\'] = tenglamalar_va_daraja(np.random.default_rng(13), 10 if a.tez else 50)\n    print("2. Kub testlari..."); out[\'kub\'] = kub_testlari(np.random.default_rng(21), dlar=(2, 4, 6, 8) if a.tez else (2, 4, 6, 8, 10, 12), raundlar=(0, 1) if a.tez else (0, 1, 2, 3))\n    print("3. Linearizatsiya hujumi..."); out[\'linearizatsiya\'] = linearizatsiya_hujumi(np.random.default_rng(31), 80, 100 if a.tez else 300)\n    E, LC = tenglamalar_soni(); out[\'tenglamalar_soni\'] = E; out[\'linearizatsiya_murakkabligi_log2\'] = LC\n    json.dump(out, open(\'natijalar/algebraik.json\', \'w\'), indent=1, ensure_ascii=False, default=list)\n    print("Saqlandi: natijalar/algebraik.json")\n',
    'boglash': '"""\nBog‘lash(holat, X) — Qo‘sh o‘rniga taklif etilayotgan yagona akslantirish (256 va 128 bitli rejimlar uchun bir xil).\nIkkala rejimda ham bayt darajasida ishlaydi: 128 bitli rejimda qo‘shni yarim baytlar juftlab baytga birlashtiriladi\n(SBZ dagi kabi), so‘ng natija yana yarim baytlarga ajratiladi.\n  Yarim qadam (T ← f(U)):  J = Σ U,  K = Π (U ∨ 1)  (mod 256)\n      aᵢ = 2·S(U[i] ⊕ X_U[i] ⊕ J) + 1,   bᵢ = 2·S(U[i+1] ⊕ X_T[i] ⊕ K) + 1\n      T[i] ← S( −aᵢ·T[i]·(bᵢ + R·T[i])⁻¹ ⊕ X_T[i] )        (R — kalitning birinchi bayti, juft)\n  Bog‘lash: o‘ng ← f(chap), chap ← f(o‘ng), o‘ng ← f(chap).\n"""\nimport numpy as np\nfrom algoritm import S, Cfg, aralash, surholat, surkalit, bzf, to_units\n\nINV = np.zeros(256, dtype=np.int64)\nfor _x in range(1, 256, 2): INV[_x] = pow(_x, -1, 256)\nSI = np.zeros(256, dtype=np.int64); SI[S] = np.arange(256)\n\ndef juftlash(u, p):      # birliklar -> baytlar\n    return u if p == 256 else 16 * u[:, 0::2] + u[:, 1::2]\ndef ajratish(b, p):      # baytlar -> birliklar\n    if p == 256: return b\n    u = np.empty((b.shape[0], 32), dtype=np.int64); u[:, 0::2] = b >> 4; u[:, 1::2] = b & 15; return u\ndef yarimlar(m):         # m = 32 yoki 16 bayt; satrda w bayt; chap yarim — satrning birinchi yarmi\n    w = 8 if m == 32 else 4\n    i = np.arange(m); return i[(i % w) < w // 2], i[(i % w) >= w // 2]\n\ndef _param(U, XT, XU):\n    J = U.sum(1) % 256; K = np.ones(U.shape[0], dtype=np.int64)\n    for i in range(U.shape[1]): K = (K * (U[:, i] | 1)) % 256\n    sig = np.roll(np.arange(U.shape[1]), -1)\n    a = (2 * S[U ^ XU ^ J[:, None]] + 1) % 256\n    b = (2 * S[U[:, sig] ^ XT ^ K[:, None]] + 1) % 256\n    return a, b\ndef _yarim(T, U, XT, XU, R):\n    a, b = _param(U, XT, XU)\n    return S[((-a * T * INV[(b + R[:, None] * T) % 256]) % 256) ^ XT]\ndef _yarim_inv(Y, U, XT, XU, R):\n    a, b = _param(U, XT, XU); f = SI[Y] ^ XT\n    return (-b * f * INV[(a + R[:, None] * f) % 256]) % 256\n\ndef boglash(holat, X, R, p):\n    """holat, X: (N,32) birliklar; R: (N,) juft bayt. Natija: (N,32) birliklar."""\n    H = juftlash(holat, p).copy(); Xb = juftlash(X, p); L, Rt = yarimlar(H.shape[1])\n    H[:, Rt] = _yarim(H[:, Rt], H[:, L], Xb[:, Rt], Xb[:, L], R)\n    H[:, L] = _yarim(H[:, L], H[:, Rt], Xb[:, L], Xb[:, Rt], R)\n    H[:, Rt] = _yarim(H[:, Rt], H[:, L], Xb[:, Rt], Xb[:, L], R)\n    return ajratish(H, p)\ndef boglash_inv(Y, X, R, p):\n    H = juftlash(Y, p).copy(); Xb = juftlash(X, p); L, Rt = yarimlar(H.shape[1])\n    H[:, Rt] = _yarim_inv(H[:, Rt], H[:, L], Xb[:, Rt], Xb[:, L], R)\n    H[:, L] = _yarim_inv(H[:, L], H[:, Rt], Xb[:, L], Xb[:, Rt], R)\n    H[:, Rt] = _yarim_inv(H[:, Rt], H[:, L], Xb[:, Rt], Xb[:, L], R)\n    return ajratish(H, p)\n\nRC = np.array([[int(S[(16 * j + i) % 256]) for i in range(32)] for j in range(11)])   # raund konstantalari\n\ndef iteration_b(holat, blok, ke, k, c, sbz=True, rounds=10, trace=None):\n    """Yangi variant: Qo‘sh -> Bog‘lash; yutish: Bog‘lash(holat, blok) [-> SBZ(holat, blok)];\n    kalit jadvali holatdan mustaqil: kₑ ← (SurKalit(kₑ) ⊕ RC_j) ∨ 1."""\n    p = c.p; R = juftlash(k, p)[:, 0] & 0xFE; m = p - 1\n    def yut(h):\n        h = boglash(h, blok, R, p)\n        return bzf(h, blok, c) if sbz else h\n    ke = ke | 1\n    holat = yut(holat)\n    for j in range(1, rounds + 1):\n        holat = aralash(holat, ke, c)\n        holat = boglash(holat, ke, R, p)\n        holat = surholat(holat); ke = (surkalit(ke) ^ (RC[j] & m)) | 1\n        holat = yut(holat) if j == 8 else boglash(holat, ke, R, p)\n        if j == 9: holat = yut(holat)\n        if trace is not None: trace.append((j, holat.copy(), ke.copy()))\n    return holat, ke\n\ndef xesh_b(msgs, keys, p, sbz=True):\n    """msgs: (N,L) bayt (L blok uzunligiga karrali), keys: (N,32 yoki 16) bayt."""\n    c = Cfg(p, \'sbz\'); N, L = msgs.shape; bl = 32 if p == 256 else 16; nb = L // bl\n    blocks = [msgs[:, i * bl:(i + 1) * bl] for i in range(nb)]\n    ny = np.zeros((N, bl), dtype=np.int64)\n    for n in range(N):\n        v = sum(int.from_bytes(bytes(b[n].tolist()), \'big\') for b in blocks) % 2 ** (8 * bl); ny[n] = list(v.to_bytes(bl, \'big\'))\n    lb = np.tile(np.array(list((L * 8).to_bytes(bl, \'big\'))), (N, 1))\n    kir = [to_units(b, p) for b in blocks] + [to_units(lb, p), to_units(ny, p)]\n    k = to_units(keys, p); ke = k.copy(); holat = kir[0]\n    for i in range(1, len(kir)): holat, ke = iteration_b(holat, kir[i], ke, k, c, sbz)\n    return holat\n',
    'yagona': '"""\nYagona (unifikatsiyalangan) yakuniy variant: ikkala rejim bayt darajasida, bir xil tuzilish.\n  256 bitli rejim: holat va kalit — 4×8 bayt; 128 bitli rejim: holat va kalit — 4×4 bayt (yarim baytlar yo‘q).\n  Aralash — diamatritsa ko‘paytmasi (mod 256): 4×8 da ikki 4×4 yarim, 4×4 da bitta matritsa.\n  Bog‘lash, SBZ, kalit jadvali, oldinga uzatish va P(H) ⊕ H — ikkala rejimda bir xil.\n  Raundlar soni R (standartda 10); yutish nuqtalari: 0, R−2, R−1 (R = 10 da 0, 8, 9).\n  Konstantalar: RCⱼ — √2 ning kasr qismi baytlari (j = 1…10, har biri 32 bayt), C₁, C₂ — √3 ning kasr qismi baytlari.\n"""\nimport numpy as np\nfrom math import isqrt\nfrom algoritm import S, dia, surholat as _surholat8, surkalit as _surkalit8\nfrom boglash import _yarim, yarimlar\n\ndef kasr_baytlar(n, N):\n    v = isqrt(n * 256 ** (2 * N)); butun = isqrt(n) * 256 ** N\n    return list((v - butun).to_bytes(N, \'big\'))\n_RC = kasr_baytlar(2, 320)\nRC = np.array([[0] * 32] + [_RC[32 * (j - 1):32 * j] for j in range(1, 11)])\n_C = kasr_baytlar(3, 64); C1 = np.array(_C[:32]) | 1; C2 = np.array(_C[32:]) | 1\n\ndef aralash_y(h, ke, w):\n    N = h.shape[0]; H = h.reshape(N, 4, w); K = ke.reshape(N, 4, w)\n    if w == 8:\n        out = np.concatenate([dia(H[:, :, :4], K[:, :, :4], 256), dia(H[:, :, 4:], K[:, :, 4:], 256)], axis=2)\n    else:\n        out = dia(H, K, 256)\n    return out.reshape(N, 4 * w) % 256\n\ndef _sur4(a, rows, cols):\n    N = a.shape[0]; m = a.reshape(N, 4, 4).copy()\n    for s_ in range(4): m[:, s_, :] = np.roll(m[:, s_, :], rows[s_], axis=1)\n    for u, n in cols.items(): m[:, :, u] = np.roll(m[:, :, u], n, axis=1)\n    return m.reshape(N, 16)\ndef surholat_y(h, w): return _surholat8(h) if w == 8 else _sur4(h, [1, 2, 3, 0], {0: 1, 1: 2, 2: 3, 3: 0})\ndef surkalit_y(k, w): return _surkalit8(k) if w == 8 else _sur4(k, [0, 3, 2, 1], {1: -3, 2: -2, 3: -1})\n\ndef boglash_y(H, X, R):\n    H = H.copy(); L, Rt = yarimlar(H.shape[1])\n    H[:, Rt] = _yarim(H[:, Rt], H[:, L], X[:, Rt], X[:, L], R)\n    H[:, L] = _yarim(H[:, L], H[:, Rt], X[:, L], X[:, Rt], R)\n    H[:, Rt] = _yarim(H[:, Rt], H[:, L], X[:, Rt], X[:, L], R)\n    return H\ndef sbz_y(h, n): return S[h ^ S[n]]\n\ndef iteration_g(holat, blok, ke, k, R_raund=10, trace=None):\n    w = holat.shape[1] // 4; Rp = k[:, 0] & 0xFE; mm = 4 * w\n    def T(nm, j):\n        if trace is not None: trace.append((nm, j, holat.copy(), ke.copy()))\n    ke = ke | 1\n    holat = sbz_y(boglash_y(holat, blok, Rp), blok); T(\'yutish0\', 0)\n    for j in range(1, R_raund + 1):\n        holat = aralash_y(holat, ke, w)\n        holat = boglash_y(holat, ke, Rp)\n        holat = surholat_y(holat, w); ke = (surkalit_y(ke, w) ^ RC[j][:mm]) | 1\n        if j == R_raund - 2: holat = sbz_y(boglash_y(holat, blok, Rp), blok)\n        else: holat = boglash_y(holat, ke, Rp)\n        if j == R_raund - 1: holat = sbz_y(boglash_y(holat, blok, Rp), blok)\n        T(\'raund\', j)\n    return holat, ke\n\ndef xesh_g(msgs, keys, R_raund=10):\n    """msgs: (N, L) bayt, L — blok uzunligiga (32 yoki 16) karrali; keys: (N, 32 yoki 16) bayt; natija (N, bl) bayt."""\n    bl = keys.shape[1]; N, L = msgs.shape; nb = L // bl\n    blocks = [msgs[:, i * bl:(i + 1) * bl] for i in range(nb)]\n    ny = np.zeros((N, bl), dtype=np.int64)\n    for n in range(N): ny[n] = list((sum(int.from_bytes(bytes(b[n].tolist()), \'big\') for b in blocks) % 2 ** (8 * bl)).to_bytes(bl, \'big\'))\n    lb = np.tile(np.array(list((L * 8).to_bytes(bl, \'big\'))), (N, 1))\n    kir = blocks + [lb, ny]; ke = keys.copy(); h = kir[0].copy()\n    for i in range(1, len(kir)):\n        h0 = h; h, ke = iteration_g(h, kir[i], ke, keys, R_raund); h = h ^ h0\n    Rc = np.full(N, 0xC8)\n    P = boglash_y(boglash_y(h, np.tile(C1[:bl], (N, 1)), Rc), np.tile(C2[:bl], (N, 1)), Rc)\n    return P ^ h\n\ndef xesh_bayt_g(xabar: bytes, kalit: bytes, bl=32, R_raund=10) -> bytes:\n    m = xabar + b"\\x00" * (-len(xabar) % bl) if xabar else b"\\x00" * bl\n    return bytes(int(v) for v in xesh_g(np.array([list(m)]), np.array([list(kalit[:bl])]), R_raund)[0])\n',
    'chegaralar': '"""\nBog‘lash orqali faol S-box larning minimal sonini hisoblash (differensial va chiziqli chegaralar).\nDifferensial model (bayt darajasidagi qisqartirilgan ayirmalar, bitta kalit — X da ayirma yo‘q):\n  yarim qadam T ← f(U): aᵢ uchun S faol, agar (i ∈ A_U) yoki ΔJ ≠ 0; bᵢ uchun S faol, agar (i+1) ∈ A_U (ΔK = 0 tanlanadi);\n  |A_U| = 1 bo‘lsa ΔJ ≠ 0 majburiy (bitta baytning modul ayirmasi nol emas), |A_U| ≥ 2 da ΔJ = 0 tanlanishi mumkin;\n  parametrlari o‘zgarmagan element: chiqish S faol ⇔ T[i] faol; parametrlari o‘zgargan element: chiqish faol yoki nofaol (buzg‘unchi tanlaydi).\n  Har bir faol S-box: p ≤ 2⁻⁶. Chiziqli model: niqob faqat T elementlari orqali o‘tkaziladi (eng arzon yo‘l), har bir faol S-box: |c| ≤ 2⁻³.\n"""\nimport numpy as np, itertools, json\n\ndef yarim_qadam(AU, AT, m):\n    """AU, AT: bitmaskalar (m bit). Natija: [(narx, yangi_AT), ...] — buzg‘unchi tanlashi mumkin bo‘lgan barcha variantlar."""\n    k = bin(AU).count(\'1\'); full = (1 << m) - 1\n    rot = lambda x: ((x >> 1) | ((x & 1) << (m - 1)))          # b_i U[i+1] ga bog‘liq: i+1 ∈ A_U ⇔ i ∈ rot(AU)\n    variantlar = []\n    for dJ in ([True] if k == 1 else ([False, True] if k >= 2 else [False])):\n        Pa = full if dJ else AU                                   # a parametri o‘zgargan elementlar\n        Pb = rot(AU)\n        nS = bin(Pa).count(\'1\') + bin(Pb).count(\'1\')\n        P = Pa | Pb\n        majburiy = AT & ~P                                        # parametr o‘zgarmagan, T faol -> chiqish faol\n        ixtiyoriy = P                                             # parametr o‘zgargan -> buzg‘unchi tanlaydi\n        variantlar.append((nS, majburiy, ixtiyoriy))\n    return variantlar\n\ndef min_chaqiruv(m, maxw=None, maxsub=None):\n    """Bitta Bog‘lash chaqiruvidagi faol S-box larning minimal soni (barcha noldan farqli kirishlar bo‘yicha).\n    Dinamik dasturlash: holat = (A_L, A_R) bitmaskalar; uchta yarim qadam."""\n    full = (1 << m) - 1; INF = 10 ** 9\n    def qadam(dist, nishon_ong):\n        new = {}\n        for (AL, AR), c in dist.items():\n            AU, AT = (AL, AR) if nishon_ong else (AR, AL)\n            for nS, maj, ixt in yarim_qadam(AU, AT, m):\n                if maxsub is None:\n                    subs = []; sub = ixt\n                    while True:\n                        subs.append(sub)\n                        if sub == 0: break\n                        sub = (sub - 1) & ixt\n                else:                                             # faqat kichik qism to‘plamlar (≤ maxsub element)\n                    bits = [1 << i for i in range(m) if (ixt >> i) & 1]\n                    subs = [sum(c_) for w in range(maxsub + 1) for c_ in itertools.combinations(bits, w)]\n                for sub in subs:                                  # ixtiyoriy elementlarning qism to‘plamlari\n                    nAT = maj | sub\n                    cost = c + nS + bin(nAT).count(\'1\')\n                    key = (AL, nAT) if nishon_ong else (nAT, AR)\n                    if cost < new.get(key, INF): new[key] = cost\n        return new\n    start = {}\n    if maxw is None:\n        for AL in range(1 << m):\n            for AR in range(1 << m):\n                if AL or AR: start[(AL, AR)] = 0\n    else:\n        for w in range(1, maxw + 1):\n            for pos in itertools.combinations(range(2 * m), w):\n                AL = sum(1 << q for q in pos if q < m); AR = sum(1 << (q - m) for q in pos if q >= m); start[(AL, AR)] = 0\n    d = qadam(start, True); d = qadam(d, False); d = qadam(d, True)\n    d = {k: v for k, v in d.items() if k != (0, 0)}\n    return min(d.values())\n\n# ---------------- Chiziqli: niqoblar zanjiri (0-qatlam modeli, Aralash toq kₑ bilan, raundlar bo‘yicha) ----------------\ndef chiziqli_zanjir(rejim=256, raundlar=4, maxw=2):\n    """Niqob bayt to‘plami sifatida. Bog‘lash narxi: o‘ng yarim bayti 2 ta (3- va 1-yarim qadam), chap yarim bayti 1 ta faol S.\n    Aralash: 0-qatlam bo‘yicha M(1)ᵀ, SurHolat — o‘rin almashtirish. 128 bitli rejimda Aralash yarim baytlarda, Bog‘lash baytlarda."""\n    from komponentlar import M_aralash, P_SUR\n    M = M_aralash(np.ones(32, dtype=np.int64)) % 2; P = P_SUR % 2\n    if rejim == 256:\n        bayt = lambda e: e; nb = 32; chap = lambda j: (j % 8) < 4\n    else:\n        bayt = lambda e: e // 2; nb = 16; chap = lambda j: (j % 4) < 2\n    def narx(v):          # v: 32 ta element bo‘yicha niqob (0/1)\n        B = set(bayt(e) for e in np.nonzero(v)[0]); return sum(1 if chap(j) else 2 for j in B)\n    best = {r: 10 ** 9 for r in range(1, raundlar + 1)}\n    for w in range(1, maxw + 1):\n        for pos in itertools.combinations(range(32), w):\n            v = np.zeros(32, dtype=np.int64); v[list(pos)] = 1; tot = 0\n            for r in range(1, raundlar + 1):           # oxiridan orqaga: Bog2, SurHolat⁻¹, Bog1, Aralashᵀ\n                tot += narx(v); v = (P.T @ v) % 2; tot += narx(v); best[r] = min(best[r], tot); v = (M.T @ v) % 2\n    return best\n\ndef min_chaqiruv_olcham(m):\n    """O‘lchamlar bo‘yicha (ruxsat etilgan barcha variantlarning ustki to‘plami) — har qanday m uchun isbotlangan quyi chegara.\n    U yarmida k ta faol bayt: parametrlar S-boxlari ≥ s(k) (s(0)=0, s(1)=m, s(k≥2)=2k), o‘zgargan parametrli elementlar ≤ |P|\n    (|P| ≤ m agar k=1, aks holda ≤ 2k); yangi T o‘lchami [max(0, t−|P|), min(m, t+|P|)] oralig‘ida (buzg‘unchi tanlaydi)."""\n    INF = 10 ** 9\n    s = lambda k: 0 if k == 0 else (m if k == 1 else 2 * k)\n    Pm = lambda k: 0 if k == 0 else (m if k == 1 else min(m, 2 * k))\n    def qadam(dist, ong):\n        new = {}\n        for (l, r), c in dist.items():\n            k, t = (l, r) if ong else (r, l)\n            lo, hi = max(0, t - Pm(k)), min(m, t + Pm(k))\n            for nt in range(lo, hi + 1):\n                cost = c + s(k) + nt; key = (l, nt) if ong else (nt, r)\n                if cost < new.get(key, INF): new[key] = cost\n        return new\n    d = {(l, r): 0 for l in range(m + 1) for r in range(m + 1) if l or r}\n    d = qadam(d, True); d = qadam(d, False); d = qadam(d, True)\n    return min(v for k, v in d.items() if k != (0, 0))\n\ndef chiziqli_zanjir_yagona(bl=32, raundlar=6, maxw=2):\n    """Yagona variant uchun chiziqli zanjir (bayt darajasi): bl = 32 — 4×8 (ikki 4×4 Aralash), bl = 16 — 4×4."""\n    from algoritm import dia\n    if bl == 32:\n        from komponentlar import M_aralash, P_SUR\n        M = M_aralash(np.ones(32, dtype=np.int64)) % 2; P = P_SUR % 2; chap = lambda j: (j % 8) < 4\n    else:\n        from yagona import _sur4\n        E = np.eye(16, dtype=np.int64).reshape(16, 4, 4); M = (dia(E, np.ones((16, 4, 4), dtype=np.int64), 2).reshape(16, 16).T) % 2\n        P = np.zeros((16, 16), dtype=np.int64)\n        for i in range(16):\n            v = np.zeros((1, 16), dtype=np.int64); v[0, i] = 1; P[:, i] = _sur4(v, [1, 2, 3, 0], {0: 1, 1: 2, 2: 3, 3: 0})[0]\n        chap = lambda j: (j % 4) < 2\n    narx = lambda v: sum(1 if chap(j) else 2 for j in np.nonzero(v)[0])\n    best = {r: 10 ** 9 for r in range(1, raundlar + 1)}\n    for w in range(1, maxw + 1):\n        for pos in itertools.combinations(range(bl), w):\n            v = np.zeros(bl, dtype=np.int64); v[list(pos)] = 1; tot = 0\n            for r in range(1, raundlar + 1):\n                tot += narx(v); v = (P.T @ v) % 2; tot += narx(v); best[r] = min(best[r], tot); v = (M.T @ v) % 2\n    return best\n',
}
for _nom, _matn in _MODULLAR.items():
    if importlib.util.find_spec(_nom) is not None and _nom not in sys.modules:
        print(f"{_nom}.py — papkadagi fayldan olinadi"); continue
    _m = types.ModuleType(_nom); _m.__file__ = f"<{_nom}>"
    sys.modules[_nom] = _m; exec(compile(_matn, f"<{_nom}>", "exec"), _m.__dict__)
    print(f"{_nom} — daftar ichidan yuklandi")


algoritm — daftar ichidan yuklandi
komponentlar — daftar ichidan yuklandi
algebraik — daftar ichidan yuklandi
boglash — daftar ichidan yuklandi
yagona — daftar ichidan yuklandi
chegaralar — daftar ichidan yuklandi


In [2]:
import numpy as np, json, os, sys, time, shutil, subprocess
from algoritm import KAT_XABAR, KAT_KALIT
from yagona import xesh_g, xesh_bayt_g, iteration_g, RC, C1, C2
from chegaralar import min_chaqiruv, min_chaqiruv_olcham, chiziqli_zanjir_yagona
from komponentlar import rank2
NATIJA = {}
print("RC₁ =", bytes(RC[1][:16].tolist()).hex(), " C₁ =", bytes(C1[:16].tolist()).hex())
NATIJA['KAT'] = {f'{bl}_{R}': xesh_bayt_g(KAT_XABAR, KAT_KALIT, bl, R).hex() for bl in (32, 16) for R in (10, 6, 5, 4)}
for k, v in NATIJA['KAT'].items(): print(f"n = {256 if k.startswith('32') else 128}, R = {k.split('_')[1]:>2s}: {v}")

RC₁ = 6a09e667f3bcc908b2fb1366ea957d3e  C₁ = bb67af8585cba73b25752d7179b93b89
n = 256, R = 10: a1073d266484de729ed9310e32affa697d159b53b078c0468e19611edd13ed37
n = 256, R =  6: 9ab07b8d692c955e67e4e60362a929d516848f3b5733cab7a24aef26b2afade7
n = 256, R =  5: 0c9baec600b8c5709298c8f789d679d68cc394fa4c531c44578bd394aa1cc6fb
n = 256, R =  4: 88e1651042acf0cffa6aeeba118beab47754084873799ed230186d0f0119cc72
n = 128, R = 10: 3f15f40025eaf5e4e1945b29dc265345
n = 128, R =  6: f84f8c106ba8ecfcf00188b0a24720f0
n = 128, R =  5: 29522bb4640b8d4e2f3d0edf0494aefe
n = 128, R =  4: f15b2677e0856c419a6b3655c918dfc4


## 1. Faol S-box larning minimal soni (differensial va chiziqli chegaralar)

In [3]:
t0 = time.time(); d8 = min_chaqiruv(8); print(f"Differensial, bitta Bog‘lash chaqiruvi, 128 bitli rejim (to‘liq qidiruv): {d8} ta faol S-box  ({time.time()-t0:.0f} s)")
for m in (8, 16): print(f"O‘lchamlar modeli, m = {m}: {min_chaqiruv_olcham(m)} ta faol S-box")
print("=> bir raundda ≥ 16 ta faol S-box, p ≤ 2^-96 (ikkala rejim)")
NATIJA['lin'] = {}
for bl in (32, 16):
    z = chiziqli_zanjir_yagona(bl, 6, 2 if bl == 32 else 3); NATIJA['lin'][bl] = z
    print(f"Chiziqli, n = {256 if bl == 32 else 128}: raundlar bo‘yicha faol S-box lar {z}  ->  |c| ≤ 2^-(3·N)")

Differensial, bitta Bog‘lash chaqiruvi, 128 bitli rejim (to‘liq qidiruv): 8 ta faol S-box  (18 s)
O‘lchamlar modeli, m = 8: 8 ta faol S-box
O‘lchamlar modeli, m = 16: 8 ta faol S-box
=> bir raundda ≥ 16 ta faol S-box, p ≤ 2^-96 (ikkala rejim)
Chiziqli, n = 256: raundlar bo‘yicha faol S-box lar {1: 2, 2: 10, 3: 26, 4: 55, 5: 101, 6: 129}  ->  |c| ≤ 2^-(3·N)
Chiziqli, n = 128: raundlar bo‘yicha faol S-box lar {1: 2, 2: 9, 3: 20, 4: 38, 5: 45, 6: 64}  ->  |c| ≤ 2^-(3·N)


## 2. Raundlar soni: statistik sinovlar

In [4]:
rng = np.random.default_rng(31)
def bitsb(a): return ((a[:, :, None] >> np.arange(8)) & 1).reshape(len(a), -1)
for bl in (32, 16):
    for R in (4, 6, 10):
        N = 200; m = rng.integers(0, 256, (N, 2 * bl)); k = rng.integers(0, 256, (N, bl)); k[:, 0] &= 0xFE
        o = xesh_g(m, k, R); m2 = m.copy(); pos = rng.integers(0, 2 * bl * 8, N); m2[np.arange(N), pos // 8] ^= (1 << (pos % 8))
        hd = bitsb(o ^ xesh_g(m2, k, R)).sum(1)
        kk = np.tile(k[0], (200, 1)); oo = xesh_g(rng.integers(0, 256, (200, 2 * bl)), kk, R) % 2
        print(f"n = {256 if bl == 32 else 128}, R = {R:2d}: o‘zgargan bitlar o‘rtacha {hd.mean():.1f} (ideal {4*bl}), min {hd.min()}, max {hd.max()};  0-qatlam rangi {rank2((oo[1:]^oo[0]).T.copy())}")

n = 256, R =  4: o‘zgargan bitlar o‘rtacha 127.8 (ideal 128), min 107, max 148;  0-qatlam rangi 32
n = 256, R =  6: o‘zgargan bitlar o‘rtacha 128.1 (ideal 128), min 105, max 153;  0-qatlam rangi 32
n = 256, R = 10: o‘zgargan bitlar o‘rtacha 128.1 (ideal 128), min 106, max 149;  0-qatlam rangi 32
n = 128, R =  4: o‘zgargan bitlar o‘rtacha 63.9 (ideal 64), min 50, max 80;  0-qatlam rangi 16
n = 128, R =  6: o‘zgargan bitlar o‘rtacha 64.2 (ideal 64), min 45, max 81;  0-qatlam rangi 16
n = 128, R = 10: o‘zgargan bitlar o‘rtacha 64.3 (ideal 64), min 48, max 82;  0-qatlam rangi 16


## 3. Tezlik (C realizatsiyasi): standart va yagona variant

In [5]:
C_MANBA = '/* O\'zMSt 285:2024, 1-algoritm: C realizatsiyasi (tezlikni o\'lchash uchun).\n   Variantlar: p = 256 yoki 16; bz = 0 (BaytZichlash) yoki 1 (SBZ);\n   toq = 1 — barcha kₑ elementlari toq (2-tavsiya); sqatlam = 1 — raund ichidagi S-box qatlami (3-tavsiya). */\n#include <stdint.h>\n#include <string.h>\n#include <stdio.h>\n#include <stdlib.h>\n#include <time.h>\n\nstatic const uint8_t ZICH[256] = {\n 5,13,6,11,1,10,15,8,0,4,7,9,2,12,3,14, 8,7,2,14,15,3,11,6,1,12,13,10,5,4,9,0,\n 14,2,13,4,12,7,1,11,6,9,0,5,3,10,8,15, 0,14,9,12,3,13,7,4,15,6,5,1,11,2,10,8,\n 3,10,7,2,4,12,9,1,14,13,15,8,0,5,11,6, 2,3,1,8,0,14,5,9,12,11,6,7,10,15,13,4,\n 10,4,14,15,9,5,8,2,11,0,1,3,12,6,7,13, 11,9,10,1,6,4,13,15,3,5,14,0,8,7,2,12,\n 1,0,3,7,13,11,10,12,9,14,4,6,15,8,5,2, 4,8,11,9,14,6,2,5,10,3,12,15,7,13,0,1,\n 9,12,15,0,2,1,14,10,5,8,11,13,4,3,6,7, 6,11,8,13,7,9,0,3,4,15,10,2,14,1,12,5,\n 15,1,0,5,10,8,3,7,13,2,9,12,6,14,4,11, 12,5,4,10,11,2,6,13,8,7,3,14,1,0,15,9,\n 7,15,12,6,5,0,4,14,2,10,8,11,13,9,1,3, 13,6,5,3,8,15,12,0,7,1,2,4,9,11,14,10};\nstatic uint8_t SB[256];\nstatic const uint8_t PRESENT[16] = {0xC,5,6,0xB,9,0,0xA,0xD,3,0xE,0xF,8,4,7,1,2};\nstatic uint32_t INV256[256], INV16[16];\n\nstatic void init_tables(void) {\n    uint8_t p = 1, q = 1;\n    do {\n        p = p ^ (uint8_t)(p << 1) ^ ((p & 0x80) ? 0x1B : 0);\n        q ^= q << 1; q ^= q << 2; q ^= q << 4; if (q & 0x80) q ^= 0x09;\n        uint8_t x = q ^ (uint8_t)((q << 1) | (q >> 7)) ^ (uint8_t)((q << 2) | (q >> 6)) ^ (uint8_t)((q << 3) | (q >> 5)) ^ (uint8_t)((q << 4) | (q >> 4));\n        SB[p] = x ^ 0x63;\n    } while (p != 1);\n    SB[0] = 0x63;\n    for (uint32_t x = 1; x < 256; x += 2) for (uint32_t y = 1; y < 256; y += 2) if (((x * y) & 255) == 1) INV256[x] = y;\n    for (uint32_t x = 1; x < 16; x += 2) for (uint32_t y = 1; y < 16; y += 2) if (((x * y) & 15) == 1) INV16[x] = y;\n}\n\ntypedef struct { uint32_t p, m; const uint32_t *inv; int bz, toq, sq; } cfg_t;\n\nstatic inline uint32_t step(uint32_t x, uint32_t A, uint32_t B, uint32_t R, const cfg_t *c) {\n    uint32_t d = (B + R * x) & c->m;\n    return (0u - A * x * c->inv[d]) & c->m;\n}\nstatic inline uint32_t prododd(const uint32_t *a, const cfg_t *c) {\n    uint32_t r = 1; for (int i = 0; i < 32; i++) r = (r * (a[i] | 1)) & c->m; return r;\n}\nstatic inline uint32_t dK(uint32_t P, uint32_t Jo, const cfg_t *c) {\n    uint32_t t = (Jo + 1) & c->m; return (P == t || P == ((0u - t) & c->m)) ? 10 : 0;\n}\nstatic void qosh(uint32_t *h, uint32_t *hn, uint32_t R, const cfg_t *c) {\n    uint32_t m = c->m, J1h = 0, J1hn = 0, h1[32], hn1[32];\n    for (int i = 0; i < 32; i++) { J1h += h[i]; J1hn += hn[i]; }\n    J1h &= m; J1hn &= m;\n    uint32_t Ph = prododd(h, c), Phn = prododd(hn, c);\n    uint32_t K1h = (Ph + dK(Ph, J1hn, c)) & m, K1hn = (Phn + dK(Phn, J1h, c)) & m;\n    uint32_t dAh = (J1hn & 1) ? 0 : 1, dAhn = (J1h & 1) ? 0 : 1;\n    for (int i = 0; i < 32; i++) {\n        h1[i]  = step(h[i],  (J1hn + 2 * hn[i] + dAh) & m, (K1h - 2 * h[i]) & m, R, c);\n        hn1[i] = step(hn[i], (J1h + 2 * h[i] + dAhn) & m, (K1hn - 2 * hn[i]) & m, R, c);\n    }\n    uint32_t J2h = 0, J2hn = 0;\n    for (int i = 0; i < 32; i++) { J2h += h1[i]; J2hn += hn1[i]; }\n    J2h &= m; J2hn &= m;\n    uint32_t P2h = prododd(h1, c), P2hn = prododd(hn1, c);\n    uint32_t K2h = (P2h + dK(P2h, J2hn, c)) & m, K2hn = (P2hn + dK(P2hn, J2h, c)) & m;\n    uint32_t dA2h = ((J1hn + J2hn) & 1) ? 0 : 1, dA2hn = ((J1h + J2h) & 1) ? 0 : 1;\n    uint32_t KKh = (K1h * K2h) & m, KKhn = (K1hn * K2hn) & m;\n    for (int i = 0; i < 32; i++) {\n        uint32_t A2 = (J1hn + J2hn + 2 * hn1[i] + dA2h) & m; if (!A2) A2 = 1;\n        uint32_t B2 = (KKh - 2 * h1[i]) & m;\n        uint32_t x2 = step(h1[i], A2, B2, R, c); if (!x2) x2 = (A2 + B2) & m;\n        uint32_t A2n = (J1h + J2h + 2 * h1[i] + dA2hn) & m; if (!A2n) A2n = 1;\n        uint32_t B2n = (KKhn - 2 * hn1[i]) & m;\n        uint32_t y2 = step(hn1[i], A2n, B2n, R, c); if (!y2) y2 = (A2n + B2n) & m;\n        h[i] = step(x2, A2, B2, R, c); hn[i] = step(y2, A2n, B2n, R, c);\n    }\n}\nstatic void dia(uint32_t *h, const uint32_t *k, int off, const cfg_t *c) {\n    uint32_t H[4][4], K[4][4], sk[4], sh[4], m = c->m;\n    for (int s = 0; s < 4; s++) for (int u = 0; u < 4; u++) { H[s][u] = h[8 * s + off + u]; K[s][u] = k[8 * s + off + u]; }\n    for (int u = 0; u < 4; u++) { sk[u] = K[0][u] + K[1][u] + K[2][u] + K[3][u]; sh[u] = H[0][u] + H[1][u] + H[2][u] + H[3][u]; }\n    for (int s = 0; s < 4; s++) for (int u = 0; u < 4; u++) {\n        uint32_t v;\n        if (s == u) { v = H[u][u] * sk[u]; for (int i = 0; i < 4; i++) if (i != u) v -= H[i][i] * K[i][u]; }\n        else { v = H[s][u] * sk[u] + K[s][u] * sh[u]; for (int i = 0; i < 4; i++) if (i != s && i != u) v -= H[s][i] * K[i][u]; }\n        h[8 * s + off + u] = v & m;\n    }\n}\nstatic void aralash(uint32_t *h, const uint32_t *ke, const cfg_t *c) { dia(h, ke, 0, c); dia(h, ke, 4, c); }\nstatic void shift(uint32_t *a, const int *rows, const int *cols) {\n    uint32_t t[32];\n    for (int s = 0; s < 4; s++) for (int u = 0; u < 8; u++) t[8 * s + u] = a[8 * s + ((u - rows[s]) % 8 + 8) % 8];\n    for (int u = 0; u < 8; u++) for (int s = 0; s < 4; s++) a[8 * s + u] = t[8 * (((s - cols[u]) % 4 + 4) % 4) + u];\n}\nstatic const int HR[4] = {1, 2, 3, 4}, HC[8] = {1, 2, 3, 0, 1, 2, 3, 0};\nstatic const int KR[4] = {4, 3, 2, 1}, KC[8] = {0, -3, -2, -1, 0, -3, -2, -1};\nstatic void tuz(uint32_t *ke, const uint32_t *k, const cfg_t *c) {\n    for (int i = 0; i < 32; i++) {\n        if (c->toq) { ke[i] |= 1; continue; }\n        if (ke[i] == 0 && !(k[i] & 1)) ke[i] = 2;\n        else if (ke[i] != 0 && (ke[i] & 1) == (k[i] & 1)) ;\n        else ke[i] = (ke[i] - 1) & c->m;\n    }\n}\nstatic void bzf(uint32_t *h, const uint32_t *hn, const cfg_t *c) {\n    if (c->p == 256) {\n        for (int i = 0; i < 32; i++) h[i] = c->bz ? SB[h[i] ^ SB[hn[i]]] : (uint32_t)((ZICH[h[i]] << 4) | ZICH[hn[i]]);\n    } else {\n        if (!c->bz) { for (int i = 0; i < 32; i++) h[i] ^= hn[i]; return; }\n        for (int j = 0; j < 16; j++) {\n            uint32_t H = 16 * h[2 * j] + h[2 * j + 1], N = 16 * hn[2 * j] + hn[2 * j + 1], Q = SB[H ^ SB[N]];\n            h[2 * j] = Q >> 4; h[2 * j + 1] = Q & 15;\n        }\n    }\n}\nstatic void sqatlam(uint32_t *h, const cfg_t *c) {\n    if (!c->sq) return;\n    for (int i = 0; i < 32; i++) h[i] = (c->p == 256) ? SB[h[i]] : PRESENT[h[i]];\n}\nstatic void iteration(uint32_t *holat, const uint32_t *blok, uint32_t *ke, const uint32_t *k, const cfg_t *c) {\n    uint32_t R = k[0], hn[32];\n    if (c->toq) for (int i = 0; i < 32; i++) ke[i] |= 1;\n    memcpy(hn, blok, sizeof hn); qosh(holat, hn, R, c); bzf(holat, hn, c);\n    for (int j = 1; j <= 10; j++) {\n        aralash(holat, ke, c);\n        qosh(holat, ke, R, c);\n        shift(holat, HR, HC); shift(ke, KR, KC);\n        sqatlam(holat, c);\n        if (j == 8) { memcpy(hn, blok, sizeof hn); qosh(holat, hn, R, c); bzf(holat, hn, c); }\n        else qosh(holat, ke, R, c);\n        tuz(ke, k, c);\n        if (j == 9) { memcpy(hn, blok, sizeof hn); qosh(holat, hn, R, c); bzf(holat, hn, c); }\n    }\n}\nstatic void units(const uint8_t *b, int bl, uint32_t *u, int p) {\n    if (p == 256) for (int i = 0; i < 32; i++) u[i] = b[i];\n    else for (int i = 0; i < bl; i++) { u[2 * i] = b[i] >> 4; u[2 * i + 1] = b[i] & 15; }\n}\n/* xesh: xabar, uzunlik (bayt), kalit (32 yoki 16 bayt) -> chiqish (32 yoki 16 bayt) */\nvoid xesh(const uint8_t *msg, size_t len, const uint8_t *key, int p, int bz, int toq, int sq, uint8_t *out) {\n    cfg_t c = {(uint32_t)p, (uint32_t)p - 1, p == 256 ? INV256 : INV16, bz, toq, sq};\n    int bl = p == 256 ? 32 : 16;\n    size_t nb = len ? (len + bl - 1) / bl : 1;\n    uint8_t blk[32], ny[32] = {0}, lb[32] = {0};\n    uint32_t holat[32], blok[32], ke[32], k[32];\n    uint64_t bits = (uint64_t)len * 8;\n    for (int i = 0; i < 8; i++) lb[bl - 1 - i] = (uint8_t)(bits >> (8 * i));\n    units(key, bl, k, p); memcpy(ke, k, sizeof ke);\n    for (size_t b = 0; b < nb + 2; b++) {\n        if (b < nb) {\n            memset(blk, 0, sizeof blk);\n            size_t n = (b + 1) * bl <= len ? (size_t)bl : (len > b * bl ? len - b * bl : 0);\n            if (n) memcpy(blk, msg + b * bl, n);\n            unsigned carry = 0;\n            for (int i = bl - 1; i >= 0; i--) { unsigned s = ny[i] + blk[i] + carry; ny[i] = s & 255; carry = s >> 8; }\n        } else memcpy(blk, b == nb ? lb : ny, bl);\n        if (b == 0) { units(blk, bl, holat, p); continue; }\n        units(blk, bl, blok, p); iteration(holat, blok, ke, k, &c);\n    }\n    if (p == 256) for (int i = 0; i < 32; i++) out[i] = (uint8_t)holat[i];\n    else for (int i = 0; i < 16; i++) out[i] = (uint8_t)(holat[2 * i] << 4 | holat[2 * i + 1]);\n}\n/* ---------- Bog\'lash (ikkala rejim uchun yagona, bayt darajasida) ---------- */\nstatic const int L256[16] = {0,1,2,3, 8,9,10,11, 16,17,18,19, 24,25,26,27};\nstatic const int R256[16] = {4,5,6,7, 12,13,14,15, 20,21,22,23, 28,29,30,31};\nstatic const int L128[8]  = {0,1, 4,5, 8,9, 12,13};\nstatic const int R128[8]  = {2,3, 6,7, 10,11, 14,15};\nstatic inline void yarim(uint8_t *H, const int *T, const int *U, int m, const uint8_t *X, uint32_t R) {\n    uint32_t J = 0, K = 1;\n    for (int i = 0; i < m; i++) { uint32_t u = H[U[i]]; J += u; K *= (u | 1); }\n    J &= 255; K &= 255;\n    uint32_t u0 = H[U[0]];\n    for (int i = 0; i < m; i++) {\n        uint32_t ui = H[U[i]], un = (i + 1 < m) ? H[U[i + 1]] : u0, xt = X[T[i]], x = H[T[i]];\n        uint32_t a = 2u * SB[ui ^ X[U[i]] ^ J] + 1, b = 2u * SB[un ^ xt ^ K] + 1;\n        uint32_t f = (0u - a * x * INV256[(b + R * x) & 255]) & 255;\n        H[T[i]] = SB[f ^ xt];\n    }\n}\nstatic void boglash(uint32_t *h, const uint32_t *x, uint32_t R, int p) {\n    uint8_t H[32], X[32];\n    if (p == 256) {\n        for (int i = 0; i < 32; i++) { H[i] = (uint8_t)h[i]; X[i] = (uint8_t)x[i]; }\n        yarim(H, R256, L256, 16, X, R); yarim(H, L256, R256, 16, X, R); yarim(H, R256, L256, 16, X, R);\n        for (int i = 0; i < 32; i++) h[i] = H[i];\n    } else {\n        for (int j = 0; j < 16; j++) { H[j] = (uint8_t)((h[2*j] << 4) | h[2*j+1]); X[j] = (uint8_t)((x[2*j] << 4) | x[2*j+1]); }\n        yarim(H, R128, L128, 8, X, R); yarim(H, L128, R128, 8, X, R); yarim(H, R128, L128, 8, X, R);\n        for (int j = 0; j < 16; j++) { h[2*j] = H[j] >> 4; h[2*j+1] = H[j] & 15; }\n    }\n}\n\n/* ---------- Yakuniy variant ---------- */\nstatic uint8_t RCT[11][32];\nstatic uint8_t C1T[32], C2T[32];\nstatic void init_yak(void) {\n    for (int j = 0; j < 11; j++) for (int i = 0; i < 32; i++) RCT[j][i] = SB[(16 * j + i) & 255];\n    for (int i = 0; i < 32; i++) { C1T[i] = SB[(7 * i + 1) & 255] | 1; C2T[i] = SB[(11 * i + 3) & 255] | 1; }\n}\nstatic void iteration_y(uint32_t *holat, const uint32_t *blok, uint32_t *ke, const uint32_t *k, const cfg_t *c) {\n    int p = c->p; uint32_t m = c->m;\n    uint32_t R = (p == 256 ? k[0] : ((k[0] << 4) | k[1])) & 0xFE;\n    for (int i = 0; i < 32; i++) ke[i] |= 1;\n    boglash(holat, blok, R, p); bzf(holat, blok, c);\n    for (int j = 1; j <= 10; j++) {\n        aralash(holat, ke, c);\n        boglash(holat, ke, R, p);\n        shift(holat, HR, HC); shift(ke, KR, KC);\n        for (int i = 0; i < 32; i++) ke[i] = ((ke[i] ^ (RCT[j][i] & m)) | 1);\n        if (j == 8) { boglash(holat, blok, R, p); bzf(holat, blok, c); }\n        else boglash(holat, ke, R, p);\n        if (j == 9) { boglash(holat, blok, R, p); bzf(holat, blok, c); }\n    }\n}\nvoid xesh_y(const uint8_t *msg, size_t len, const uint8_t *key, int p, uint8_t *out) {\n    cfg_t c = {(uint32_t)p, (uint32_t)p - 1, p == 256 ? INV256 : INV16, 1, 0, 0};\n    int bl = p == 256 ? 32 : 16; size_t nb = len ? (len + bl - 1) / bl : 1;\n    uint8_t blk[32], ny[32] = {0}, lb[32] = {0};\n    uint32_t holat[32], blok[32], ke[32], k[32], h0[32], c1[32], c2[32], P[32];\n    uint64_t bits = (uint64_t)len * 8;\n    for (int i = 0; i < 8; i++) lb[bl - 1 - i] = (uint8_t)(bits >> (8 * i));\n    units(key, bl, k, p); memcpy(ke, k, sizeof ke);\n    for (size_t b = 0; b < nb + 2; b++) {\n        if (b < nb) {\n            memset(blk, 0, sizeof blk);\n            size_t n = (b + 1) * bl <= len ? (size_t)bl : (len > b * bl ? len - b * bl : 0);\n            if (n) memcpy(blk, msg + b * bl, n);\n            unsigned carry = 0;\n            for (int i = bl - 1; i >= 0; i--) { unsigned s = ny[i] + blk[i] + carry; ny[i] = s & 255; carry = s >> 8; }\n        } else memcpy(blk, b == nb ? lb : ny, bl);\n        if (b == 0) { units(blk, bl, holat, p); continue; }\n        units(blk, bl, blok, p); memcpy(h0, holat, sizeof h0);\n        iteration_y(holat, blok, ke, k, &c);\n        for (int i = 0; i < 32; i++) holat[i] ^= h0[i];                 /* oldinga uzatish */\n    }\n    for (int i = 0; i < 32; i++) { c1[i] = C1T[i] & c.m; c2[i] = C2T[i] & c.m; }\n    memcpy(P, holat, sizeof P); boglash(P, c1, 0xC8, p); boglash(P, c2, 0xC8, p);\n    for (int i = 0; i < 32; i++) holat[i] ^= P[i];                        /* chiqish o\'zgartirishi */\n    if (p == 256) for (int i = 0; i < 32; i++) out[i] = (uint8_t)holat[i];\n    else for (int i = 0; i < 16; i++) out[i] = (uint8_t)(holat[2 * i] << 4 | holat[2 * i + 1]);\n}\nstatic const uint8_t RCG[11][32] = {{0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0},{106,9,230,103,243,188,201,8,178,251,19,102,234,149,125,62,58,222,193,117,18,119,80,153,218,47,89,11,6,103,50,42},{149,249,6,8,117,113,69,135,81,99,252,223,185,7,182,114,30,233,80,188,135,56,246,148,240,9,14,108,123,244,78,209},{164,64,93,14,133,94,62,156,166,11,56,192,35,120,102,247,149,99,121,34,45,16,139,20,140,21,120,228,94,248,156,103},{141,171,81,71,23,111,211,185,150,84,198,134,99,231,144,155,234,94,36,31,6,220,176,93,213,73,65,19,32,129,148,149},{2,114,149,109,177,250,29,251,233,167,64,89,215,146,124,24,132,201,181,121,170,81,108,163,113,158,104,54,223,4,109,142},{2,9,184,3,252,100,106,94,102,84,189,62,247,180,61,127,237,67,124,127,148,68,38,15,189,64,196,131,239,85,3,133},{131,249,123,189,69,239,184,102,49,7,20,93,95,235,231,101,164,158,148,236,127,89,113,5,251,252,46,31,167,99,239,1},{243,89,156,130,242,254,80,11,132,140,240,189,37,42,224,70,191,159,30,247,148,125,70,118,154,248,193,75,204,103,199,194},{144,190,118,146,155,5,120,193,11,88,79,180,135,201,36,245,183,31,130,220,210,144,54,9,222,232,145,41,131,212,234,173},{14,234,50,31,116,137,244,106,126,144,48,190,32,251,118,148,239,181,140,153,132,205,215,10,29,169,4,92,61,19,58,6}};\nstatic const uint8_t C1G[32] = {187,103,175,133,133,203,167,59,37,117,45,113,121,185,59,137,37,217,53,205,83,219,71,153,199,33,167,73,111,69,167,227};\nstatic const uint8_t C2G[32] = {73,11,207,217,95,241,93,189,169,147,11,175,19,35,143,135,205,77,243,77,163,161,237,105,209,205,51,161,27,217,163,131};\n\n/* ---------- Yagona variant: ikkala rejim bayt darajasida (4×8 yoki 4×4), R raund ---------- */\nstatic void dia_b(uint8_t *h, const uint8_t *k, int w, int off) {\n    uint32_t H[4][4], K[4][4], sk[4], sh[4];\n    for (int s = 0; s < 4; s++) for (int u = 0; u < 4; u++) { H[s][u] = h[w * s + off + u]; K[s][u] = k[w * s + off + u]; }\n    for (int u = 0; u < 4; u++) { sk[u] = K[0][u] + K[1][u] + K[2][u] + K[3][u]; sh[u] = H[0][u] + H[1][u] + H[2][u] + H[3][u]; }\n    for (int s = 0; s < 4; s++) for (int u = 0; u < 4; u++) {\n        uint32_t v;\n        if (s == u) { v = H[u][u] * sk[u]; for (int i = 0; i < 4; i++) if (i != u) v -= H[i][i] * K[i][u]; }\n        else { v = H[s][u] * sk[u] + K[s][u] * sh[u]; for (int i = 0; i < 4; i++) if (i != s && i != u) v -= H[s][i] * K[i][u]; }\n        h[w * s + off + u] = (uint8_t)v;\n    }\n}\nstatic void shift_b(uint8_t *a, int w, const int *rows, const int *cols) {\n    uint8_t t[32];\n    for (int s = 0; s < 4; s++) for (int u = 0; u < w; u++) t[w * s + u] = a[w * s + (((u - rows[s]) % w) + w) % w];\n    for (int u = 0; u < w; u++) for (int s = 0; s < 4; s++) a[w * s + u] = t[w * ((((s - cols[u]) % 4) + 4) % 4) + u];\n}\nstatic const int GHR8[4] = {1,2,3,4}, GHC8[8] = {1,2,3,0,1,2,3,0}, GKR8[4] = {4,3,2,1}, GKC8[8] = {0,-3,-2,-1,0,-3,-2,-1};\nstatic const int GHR4[4] = {1,2,3,0}, GHC4[4] = {1,2,3,0}, GKR4[4] = {0,3,2,1}, GKC4[4] = {0,-3,-2,-1};\nstatic void bog_b(uint8_t *H, const uint8_t *X, uint32_t R, int m) {\n    if (m == 32) { yarim(H, R256, L256, 16, X, R); yarim(H, L256, R256, 16, X, R); yarim(H, R256, L256, 16, X, R); }\n    else { yarim(H, R128, L128, 8, X, R); yarim(H, L128, R128, 8, X, R); yarim(H, R128, L128, 8, X, R); }\n}\nstatic inline void sbz_b(uint8_t *h, const uint8_t *n, int m) { for (int i = 0; i < m; i++) h[i] = SB[h[i] ^ SB[n[i]]]; }\nstatic void iteration_g(uint8_t *h, const uint8_t *blok, uint8_t *ke, const uint8_t *k, int m, int RR) {\n    int w = m / 4; uint32_t R = k[0] & 0xFE;\n    for (int i = 0; i < m; i++) ke[i] |= 1;\n    bog_b(h, blok, R, m); sbz_b(h, blok, m);\n    for (int j = 1; j <= RR; j++) {\n        if (w == 8) { dia_b(h, ke, 8, 0); dia_b(h, ke, 8, 4); } else dia_b(h, ke, 4, 0);\n        bog_b(h, ke, R, m);\n        if (w == 8) { shift_b(h, 8, GHR8, GHC8); shift_b(ke, 8, GKR8, GKC8); } else { shift_b(h, 4, GHR4, GHC4); shift_b(ke, 4, GKR4, GKC4); }\n        for (int i = 0; i < m; i++) ke[i] = (ke[i] ^ RCG[j][i]) | 1;\n        if (j == RR - 2) { bog_b(h, blok, R, m); sbz_b(h, blok, m); }\n        else bog_b(h, ke, R, m);\n        if (j == RR - 1) { bog_b(h, blok, R, m); sbz_b(h, blok, m); }\n    }\n}\n/* bl = 32 (256 bitli) yoki 16 (128 bitli) */\nvoid xesh_g(const uint8_t *msg, size_t len, const uint8_t *key, int bl, int RR, uint8_t *out) {\n    size_t nb = len ? (len + bl - 1) / bl : 1;\n    uint8_t blk[32], ny[32] = {0}, lb[32] = {0}, h[32], ke[32], h0[32], P[32];\n    uint64_t bits = (uint64_t)len * 8;\n    for (int i = 0; i < 8; i++) lb[bl - 1 - i] = (uint8_t)(bits >> (8 * i));\n    memcpy(ke, key, bl);\n    for (size_t b = 0; b < nb + 2; b++) {\n        if (b < nb) {\n            memset(blk, 0, sizeof blk);\n            size_t n = (b + 1) * bl <= len ? (size_t)bl : (len > b * bl ? len - b * bl : 0);\n            if (n) memcpy(blk, msg + b * bl, n);\n            unsigned carry = 0;\n            for (int i = bl - 1; i >= 0; i--) { unsigned s = ny[i] + blk[i] + carry; ny[i] = s & 255; carry = s >> 8; }\n        } else memcpy(blk, b == nb ? lb : ny, bl);\n        if (b == 0) { memcpy(h, blk, bl); continue; }\n        memcpy(h0, h, bl); iteration_g(h, blk, ke, key, bl, RR);\n        for (int i = 0; i < bl; i++) h[i] ^= h0[i];\n    }\n    memcpy(P, h, bl); bog_b(P, C1G, 0xC8, bl); bog_b(P, C2G, 0xC8, bl);\n    for (int i = 0; i < bl; i++) out[i] = h[i] ^ P[i];\n}\nstatic double now(void) { return (double)clock() / CLOCKS_PER_SEC; }\nint main(int argc, char **argv) {\n    init_tables(); init_yak();\n    static const uint8_t KEY[32] = {0xc8,0x6e,0xd4,0xb2,0x75,0x6d,0xf1,0xcd,0x6b,0x6e,0xca,0x81,0xd4,0x41,0x80,0x92,\n                                    0x21,0xab,0x81,0xbb,0xa0,0xc7,0xc1,0x50,0x44,0xbe,0x8f,0xeb,0x93,0x4f,0x95,0xa2};\n    const char *M = "funksiyaxeshirovaniyainformasionnayatexnologiyakriptograficheska";\n    uint8_t out[32];\n    if (argc > 1 && !strcmp(argv[1], "katg")) {\n        int Rs[4] = {10, 6, 5, 4};\n        for (int bl = 32; bl >= 16; bl /= 2) for (int r = 0; r < 4; r++) { xesh_g((const uint8_t *)M, 64, KEY, bl, Rs[r], out);\n            printf("%d_%d ", bl, Rs[r]); for (int i = 0; i < bl; i++) printf("%02x", out[i]); printf("\\n"); }\n        return 0;\n    }\n    if (argc > 1 && !strcmp(argv[1], "tezg")) {       /* tezg bl R bayt takror */\n        int bl = atoi(argv[2]), RR = atoi(argv[3]); size_t n = (size_t)atol(argv[4]); int rep = atoi(argv[5]);\n        uint8_t *buf = malloc(n); for (size_t i = 0; i < n; i++) buf[i] = (uint8_t)(i * 131 + 7);\n        double best = 1e9;\n        for (int r = 0; r < rep; r++) { double t = now(); xesh_g(buf, n, KEY, bl, RR, out); t = now() - t; if (t < best) best = t; }\n        printf("%.6f %02x\\n", best, out[0]); free(buf); return 0;\n    }\n    if (argc > 1 && !strcmp(argv[1], "katy")) {\n        for (int p = 256; p >= 16; p /= 16) { xesh_y((const uint8_t *)M, 64, KEY, p, out);\n            printf("%d ", p); for (int i = 0; i < (p == 256 ? 32 : 16); i++) printf("%02x", out[i]); printf("\\n"); }\n        return 0;\n    }\n    if (argc > 1 && !strcmp(argv[1], "tezy")) {       /* tezy p bayt takror */\n        int p = atoi(argv[2]); size_t n = (size_t)atol(argv[3]); int rep = atoi(argv[4]);\n        uint8_t *buf = malloc(n); for (size_t i = 0; i < n; i++) buf[i] = (uint8_t)(i * 131 + 7);\n        double best = 1e9;\n        for (int r = 0; r < rep; r++) { double t = now(); xesh_y(buf, n, KEY, p, out); t = now() - t; if (t < best) best = t; }\n        printf("%.6f %02x\\n", best, out[0]); free(buf); return 0;\n    }\n    if (argc > 1 && !strcmp(argv[1], "kat")) {               /* nazorat qiymatlari */\n        for (int p = 256; p >= 16; p /= 16) for (int v = 0; v < 6; v++) {\n            int bz = v >= 1, toq = v >= 2, sq = v >= 3;\n            if (v == 4) { bz = 0; toq = 1; sq = 0; } if (v == 5) { bz = 0; toq = 0; sq = 1; }\n            xesh((const uint8_t *)M, 64, KEY, p, bz, toq, sq, out);\n            printf("%d %d %d %d ", p, bz, toq, sq); for (int i = 0; i < (p == 256 ? 32 : 16); i++) printf("%02x", out[i]); printf("\\n");\n        }\n        return 0;\n    }\n    /* tezlik: argv = p bz toq sq bayt_soni takror */\n    int p = atoi(argv[1]), bz = atoi(argv[2]), toq = atoi(argv[3]), sq = atoi(argv[4]);\n    size_t n = (size_t)atol(argv[5]); int rep = atoi(argv[6]);\n    uint8_t *buf = malloc(n); for (size_t i = 0; i < n; i++) buf[i] = (uint8_t)(i * 131 + 7);\n    double best = 1e9;\n    for (int r = 0; r < rep; r++) { double t = now(); xesh(buf, n, KEY, p, bz, toq, sq, out); t = now() - t; if (t < best) best = t; }\n    printf("%.6f %02x\\n", best, out[0]);\n    free(buf); return 0;\n}\n'

In [6]:
open('yagona.c', 'w', encoding='utf-8').write(C_MANBA)
EXE = os.path.abspath('yag.exe' if sys.platform.startswith('win') else 'yag')
CC = next((c for c in ('gcc', 'cc', 'clang') if shutil.which(c)), None); C_BOR = False
if CC:
    r = subprocess.run([CC, '-O2', '-o', EXE, 'yagona.c'], capture_output=True, text=True); C_BOR = r.returncode == 0
    print("Kompilyator:", CC, "— muvaffaqiyatli" if C_BOR else "— xato:\n" + r.stderr)
else: print("C kompilyatori topilmadi.")
if C_BOR:
    c = dict(l.split() for l in subprocess.run([EXE, 'katg'], capture_output=True, text=True).stdout.splitlines())
    print("C = Python (barcha nazorat qiymatlari):", all(c[k] == v for k, v in NATIJA['KAT'].items()))
    res = {}
    for tur in range(3):
        for bl, p in ((32, 256), (16, 16)):
            res.setdefault(f'{bl}_standart', []).append(float(subprocess.run([EXE, str(p), '0', '0', '0', '262144', '3'], capture_output=True, text=True).stdout.split()[0]))
            for R in (10, 6, 4):
                res.setdefault(f'{bl}_R{R}', []).append(float(subprocess.run([EXE, 'tezg', str(bl), str(R), '262144', '3'], capture_output=True, text=True).stdout.split()[0]))
    NATIJA['tezlik'] = {}
    for bl in (32, 16):
        a = min(res[f'{bl}_standart'])
        for k in ['standart', 'R10', 'R6', 'R4']:
            t = min(res[f'{bl}_{k}']); NATIJA['tezlik'][f'{bl}_{k}'] = round(a / t, 2)
            print(f"n = {256 if bl == 32 else 128:3d}  {k:8s} {262144/t/1e6:6.2f} MB/s  standartdan {a/t:.2f} marta tez")
json.dump(NATIJA, open('yagona_natijalari.json', 'w'), ensure_ascii=False, indent=1, default=str)
print("Natijalar yagona_natijalari.json ga yozildi.")

Kompilyator: gcc — muvaffaqiyatli
C = Python (barcha nazorat qiymatlari): True
n = 256  standart   3.57 MB/s  standartdan 1.00 marta tez
n = 256  R10        3.93 MB/s  standartdan 1.10 marta tez
n = 256  R6         6.47 MB/s  standartdan 1.82 marta tez
n = 256  R4         9.37 MB/s  standartdan 2.63 marta tez
n = 128  standart   2.05 MB/s  standartdan 1.00 marta tez
n = 128  R10        4.69 MB/s  standartdan 2.28 marta tez
n = 128  R6         6.70 MB/s  standartdan 3.27 marta tez
n = 128  R4        11.04 MB/s  standartdan 5.38 marta tez
Natijalar yagona_natijalari.json ga yozildi.
